# **FAISS Vector Search Tutorial**
## Hugging Face SentenceTransformers + LangChain

This notebook teaches FAISS from first principles and then connects it to your SQL-agent few-shot example workflow.

You will learn:
- What FAISS is and how it differs from ChromaDB
- How to create embeddings with `sentence-transformers/all-MiniLM-L6-v2`
- `IndexFlatL2` vs `IndexFlatIP`
- Cosine similarity with normalized embeddings
- Adding and searching vectors
- Saving and reloading FAISS indexes
- Why metadata must be handled separately in raw FAISS
- LangChain's FAISS wrapper
- Using FAISS to retrieve relevant SQL few-shot examples


## 1. Check installed packages

In [ ]:
import faiss
import numpy as np
import sentence_transformers

print("FAISS:", getattr(faiss, "__version__", "version not exposed"))
print("NumPy:", np.__version__)
print("SentenceTransformers:", sentence_transformers.__version__)


## 2. Example documents

FAISS stores vectors, so we will separately keep the original documents and metadata.


In [ ]:
documents = [
    "Levi sells casual white cotton t-shirts.",
    "Nike has black sports t-shirts designed for training.",
    "Adidas offers blue performance shirts for athletes.",
    "Levi white shirts are available in multiple sizes.",
    "Nike running apparel includes lightweight training tops.",
    "Discounts can reduce the final selling price of shirts."
]

metadatas = [
    {"brand": "Levi", "color": "White", "type": "product"},
    {"brand": "Nike", "color": "Black", "type": "product"},
    {"brand": "Adidas", "color": "Blue", "type": "product"},
    {"brand": "Levi", "color": "White", "type": "inventory"},
    {"brand": "Nike", "color": "Unknown", "type": "sports"},
    {"brand": "Generic", "color": "Unknown", "type": "discount"},
]

for i, doc in enumerate(documents):
    print(i, doc, metadatas[i])


## 3. Load a Hugging Face SentenceTransformer

We use `all-MiniLM-L6-v2`, which creates 384-dimensional semantic embeddings.


In [ ]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

print("Loaded:", model_name)


## 4. Generate normalized embeddings

`normalize_embeddings=True` makes each vector length approximately 1. This is useful for cosine similarity.


In [ ]:
embeddings = model.encode(
    documents,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings, dtype="float32")

print("Documents:", len(embeddings))
print("Shape:", embeddings.shape)
print("Dimensions:", embeddings.shape[1])
print("First vector, first 10 values:", embeddings[0][:10])


## 5. L2 search with `IndexFlatL2`

`IndexFlatL2` performs exact nearest-neighbor search using Euclidean distance.

Smaller distance means more similar.


In [ ]:
dimension = embeddings.shape[1]

index_l2 = faiss.IndexFlatL2(dimension)
index_l2.add(embeddings)

print("Vectors stored:", index_l2.ntotal)


In [ ]:
query = "white Levi shirts"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True
).astype("float32")

distances, indices = index_l2.search(query_embedding, k=3)

print("Distances:", distances)
print("Indices:", indices)


In [ ]:
for rank, (idx, distance) in enumerate(zip(indices[0], distances[0]), start=1):
    print(f"Rank {rank}")
    print("Document:", documents[idx])
    print("Metadata:", metadatas[idx])
    print("L2 distance:", float(distance))
    print("-" * 60)


## 6. Cosine similarity with `IndexFlatIP`

FAISS does not have an `IndexFlatCosine`.

For normalized vectors:

`cosine similarity = inner product`

So we use `IndexFlatIP`.

Larger score means more similar.


In [ ]:
index_cosine = faiss.IndexFlatIP(dimension)
index_cosine.add(embeddings)

scores, indices = index_cosine.search(query_embedding, k=3)

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    print(f"Rank {rank}")
    print("Document:", documents[idx])
    print("Metadata:", metadatas[idx])
    print("Cosine similarity:", float(score))
    print("-" * 60)


## 7. Test semantic queries

SentenceTransformer can match meaning even when the wording is different.


In [ ]:
test_queries = [
    "light-colored Levi clothing",
    "sportswear for gym training",
    "products with lower final price",
    "blue athletic clothing"
]

for q in test_queries:
    q_emb = model.encode([q], normalize_embeddings=True).astype("float32")
    scores, ids = index_cosine.search(q_emb, 2)

    print("\nQUERY:", q)
    for idx, score in zip(ids[0], scores[0]):
        print(f"{float(score):.4f} -> {documents[idx]}")


## 8. Important limitation: raw FAISS does not store your documents

FAISS stores vectors and their positions.

If FAISS returns vector position `3`, your application must know that it maps to:

```python
documents[3]
metadatas[3]
```

A common architecture is:

```text
FAISS -> vectors
SQLite/PostgreSQL/JSON/docstore -> documents + metadata
```


## 9. Add a new vector

In [ ]:
new_document = "Puma sells red casual shirts."

new_embedding = model.encode(
    [new_document],
    normalize_embeddings=True
).astype("float32")

index_cosine.add(new_embedding)

documents.append(new_document)
metadatas.append({
    "brand": "Puma",
    "color": "Red",
    "type": "product"
})

print("Vectors:", index_cosine.ntotal)
print("Documents:", len(documents))


## 10. Save and reload a FAISS index

In [ ]:
from pathlib import Path

index_path = Path("tshirt_faiss.index")

faiss.write_index(index_cosine, str(index_path))
print("Saved:", index_path.resolve())


In [ ]:
loaded_index = faiss.read_index(str(index_path))

print("Loaded vectors:", loaded_index.ntotal)
print("Dimension:", loaded_index.d)


## 11. Save documents and metadata separately

Saving a FAISS index does not save your application records.


In [ ]:
import json

records = [
    {"document": doc, "metadata": meta}
    for doc, meta in zip(documents, metadatas)
]

Path("tshirt_documents.json").write_text(
    json.dumps(records, indent=2),
    encoding="utf-8"
)

print("Saved document records.")


## 12. Metadata filtering in raw FAISS

Raw FAISS does not provide Chroma-style filtering such as:

```python
where={"brand": "Levi"}
```

A simple approach is to retrieve candidates first, then filter in Python.


In [ ]:
q = model.encode(
    ["white shirt"],
    normalize_embeddings=True
).astype("float32")

scores, ids = index_cosine.search(q, 6)

filtered = []

for idx, score in zip(ids[0], scores[0]):
    if metadatas[idx]["brand"] == "Levi":
        filtered.append({
            "document": documents[idx],
            "metadata": metadatas[idx],
            "score": float(score)
        })

filtered


## 13. Common FAISS index types

### `IndexFlatL2`
- exact search
- Euclidean distance
- no training
- simplest baseline

### `IndexFlatIP`
- exact inner-product search
- works as cosine similarity with normalized vectors

### `IndexIVFFlat`
- approximate search
- clusters vectors
- requires training
- faster for much larger datasets

### HNSW
- graph-based approximate search
- fast with strong recall
- uses more memory

For your current SQL-agent few-shot examples, `IndexFlatIP` plus normalized embeddings is a sensible starting point.


# Part 2: FAISS with LangChain

## 14. Create a LangChain-compatible SentenceTransformer embedding class

In [ ]:
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

lc_embeddings = SentenceTransformerEmbeddings(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding dimensions:", len(lc_embeddings.embed_query("white Levi shirt")))


## 15. Create LangChain documents and FAISS vector store

In [ ]:
lc_documents = [
    Document(page_content=doc, metadata=meta)
    for doc, meta in zip(documents, metadatas)
]

vectorstore = FAISS.from_documents(
    lc_documents,
    lc_embeddings
)

print("Vectors:", vectorstore.index.ntotal)


## 16. LangChain similarity search

In [ ]:
results = vectorstore.similarity_search(
    "white Levi clothes",
    k=3
)

for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print("Text:", doc.page_content)
    print("Metadata:", doc.metadata)
    print("-" * 60)


## 17. Similarity search with scores

In [ ]:
results = vectorstore.similarity_search_with_score(
    "Nike training apparel",
    k=3
)

for doc, score in results:
    print("Text:", doc.page_content)
    print("Score:", score)
    print("-" * 60)


## 18. Metadata filtering through the LangChain wrapper

In [ ]:
filtered = vectorstore.similarity_search(
    "white shirt",
    k=3,
    filter={"brand": "Levi"}
)

for doc in filtered:
    print(doc.page_content, doc.metadata)


## 19. Turn FAISS into a retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

retrieved = retriever.invoke(
    "athletic clothes for exercise"
)

for doc in retrieved:
    print(doc.page_content)


## 20. Save and reload LangChain FAISS

In [ ]:
save_directory = "langchain_faiss_tshirts"

vectorstore.save_local(save_directory)

loaded_vectorstore = FAISS.load_local(
    save_directory,
    lc_embeddings,
    allow_dangerous_deserialization=True
)

print("Loaded vectors:", loaded_vectorstore.index.ntotal)


`allow_dangerous_deserialization=True` should only be used for FAISS stores you trust. Do not deserialize random downloaded files.


# Part 3: FAISS for your SQL agent

Now we use FAISS to select relevant few-shot SQL examples before building a prompt for the LLM.


In [ ]:
sql_examples = [
    {
        "Question": "How many white Levi shirts are in stock?",
        "SQLQuery": "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand='Levi' AND color='White';",
        "SQLResult": "[(202,)]",
        "Answer": "There are 202 white Levi shirts in stock."
    },
    {
        "Question": "How many Nike product variants exist?",
        "SQLQuery": "SELECT COUNT(*) FROM t_shirts WHERE brand='Nike';",
        "SQLResult": "[(12,)]",
        "Answer": "There are 12 Nike product variants."
    },
    {
        "Question": "What is the total stock of Adidas shirts?",
        "SQLQuery": "SELECT SUM(stock_quantity) FROM t_shirts WHERE brand='Adidas';",
        "SQLResult": "[(350,)]",
        "Answer": "There are 350 Adidas shirts in stock."
    },
    {
        "Question": "What is the average price of Levi shirts?",
        "SQLQuery": "SELECT AVG(price) FROM t_shirts WHERE brand='Levi';",
        "SQLResult": "[(25.50,)]",
        "Answer": "The average Levi shirt price is 25.50."
    },
    {
        "Question": "How many different shirt sizes are available?",
        "SQLQuery": "SELECT COUNT(DISTINCT size) FROM t_shirts;",
        "SQLResult": "[(4,)]",
        "Answer": "There are 4 different shirt sizes."
    }
]


## 21. Store SQL examples in FAISS

In [ ]:
sql_documents = [
    Document(
        page_content=example["Question"],
        metadata={
            "SQLQuery": example["SQLQuery"],
            "SQLResult": example["SQLResult"],
            "Answer": example["Answer"]
        }
    )
    for example in sql_examples
]

sql_vectorstore = FAISS.from_documents(
    sql_documents,
    lc_embeddings
)

print("SQL example vectors:", sql_vectorstore.index.ntotal)


## 22. Retrieve semantically relevant examples

In [ ]:
new_question = "How many black Nike shirts do I have?"

similar_examples = sql_vectorstore.similarity_search(
    new_question,
    k=2
)

for i, doc in enumerate(similar_examples, start=1):
    print(f"Selected example {i}")
    print("Question:", doc.page_content)
    print("SQLQuery:", doc.metadata["SQLQuery"])
    print("Answer:", doc.metadata["Answer"])
    print("-" * 70)


This is directly relevant to the earlier `COUNT(...)` versus `SUM(stock_quantity)` problem.

The purpose of semantic few-shot retrieval is to show the LLM examples with the same **business meaning**, not merely similar SQL syntax.


# FAISS vs ChromaDB

| Feature | FAISS | ChromaDB |
|---|---|---|
| Vector search | Excellent | Excellent |
| Exact/ANN index control | Excellent | More abstracted |
| Documents built in | No | Yes |
| Metadata built in | No | Yes |
| Native metadata filtering | No | Yes |
| Persistence | Index files | Database-style |
| LangChain integration | Excellent | Excellent |
| Easy RAG prototyping | Good | Very easy |
| Low-level vector research | Excellent | Less direct |

For SQL few-shot selection, FAISS is perfectly suitable.

For a richer knowledge base where you frequently filter records by metadata, ChromaDB is generally easier.


# Exercises

1. Add a new document: `Adidas black shirts are designed for football practice.`
2. Search for `dark Adidas sportswear`.
3. Compare `IndexFlatL2` and `IndexFlatIP`.
4. Save the FAISS index plus documents and metadata, restart the kernel, and restore all three.
5. Add a SQL example using `COUNT(DISTINCT size)` and test whether FAISS retrieves it for a semantically similar question.
6. Expand the SQL few-shot set to at least 20 examples and compare `k=1`, `k=2`, `k=3`, and `k=5`.


# Final mental model

```text
Text
  ↓
SentenceTransformer
  ↓
384-dimensional normalized vector
  ↓
FAISS index
  ↓
nearest vector positions
  ↓
documents / metadata
```

With LangChain:

```text
User question
  ↓
Embedding model
  ↓
FAISS
  ↓
Relevant Documents
  ↓
Retriever / FewShotPromptTemplate / Agent
```

The key point is that FAISS is best understood as a **high-performance vector indexing and similarity-search library**, not a full document database.
